# 07 Player Case Study Generator

This coach-analysis notebook translates defensive-action data, tactical proxies and out-of-fold model predictions into football language. It writes figures to `outputs/coach_analysis/figures/`, tables to `outputs/coach_analysis/tables/`, and video-review candidates to `outputs/coach_analysis/video_review/`.

## Coach’s question

> Which player situations are worth video review without turning exploratory signals into a ranking?

## Match situation

Recognisable situation: the defending team has just made a defensive action and the coach needs to know whether the attack is controlled, delayed, or still dangerous.

## What the current data can measure

Observed outcomes, locations, event types, tactical phase proxies, selected 360 visibility/spatial features where present, out-of-fold expected shot probability and expected future xG.

## What it cannot measure

Body orientation, exact run trajectories, intentions, responsibility assignment, off-camera players and full option availability are unmeasured unless explicit local columns exist.

## Data population and filters

In [ ]:

from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'requirements.txt').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
NOTEBOOK_NAME = '07_player_case_study_generator'
from dax.coach_analysis.loaders import validate_inputs, read_optional_table, join_oof, ensure_output_dirs
from dax.coach_analysis.populations import box_defence_population, wide_defence_population, transition_population, apply_visibility_filter
from dax.coach_analysis.sequences import construct_sequences
from dax.coach_analysis.metrics import add_suppression, summary_table
from dax.coach_analysis.comparisons import competition_comparison
from dax.coach_analysis.representative_events import select_representative_events
from dax.coach_analysis.labels import label_box_defence, label_rules
from dax.coach_analysis.plotting import action_pitch_map
import pandas as pd, numpy as np
SEED=7
outputs=ensure_output_dirs(ROOT)
status=validate_inputs(ROOT)
display(status)
actions=read_optional_table('data/features/player_defensive_actions.parquet', ROOT)
cls=read_optional_table('outputs/oof/classification_oof.parquet', ROOT)
reg=read_optional_table('outputs/oof/regression_oof.parquet', ROOT)
two=read_optional_table('outputs/oof/two_part_future_xg_oof.parquet', ROOT)
if actions.empty:
    print('Missing player_defensive_actions.parquet: notebook renders readiness guidance but cannot compute football findings locally.')
else:
    actions=add_suppression(join_oof(actions, cls, reg, two))
    actions=construct_sequences(actions)
    population=actions
    if '01' in NOTEBOOK_NAME or '05' in NOTEBOOK_NAME:
        population=label_box_defence(population)
    print('Population rows:', len(population))
    groups=[c for c in ['coach_tactical_label','coach_pitch_zone','action_family','event_type','phase','competition','position_group'] if c in population.columns][:2] or [population.columns[0]]
    table=summary_table(population, groups).head(20)
    display(table)
    table.to_csv(outputs['tables'] / f'07_player_case_study_generator_summary.csv', index=False)
    video=select_representative_events(population, reason='candidate event for coach video review')
    display(video)
    video.to_csv(outputs['video_review'] / f'07_player_case_study_generator_candidate_events.csv', index=False)
    fig,_=action_pitch_map(population.head(1000), '07_player_case_study_generator defensive action map', outputs['figures'] / f'07_player_case_study_generator_pitch_map.png')


## Descriptive evidence

## Model-based evidence

Expected shot probability means the model-estimated chance of a shot in the short horizon. Expected future xG means model-estimated shot quality allowed in that horizon. Suppression is expected minus observed, so positive values indicate less observed danger than expected and require video confirmation.

## Tactical interpretation

## Representative situations for video review

Candidate events are selected to guide analyst review; they do not explain why the event occurred.

## Coaching takeaways

## Limitations

## Follow-up question